What is the coverage of the data  on pools?
number of pools per state and constituency?
number of reviews per pools available in the dataset 

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import os



In [ ]:
import matplotlib
print(matplotlib.__version__)

In [ ]:
import matplotlib.pyplot as plt
import os
import functools

# Create folder
save_dir = "/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/visuals"
os.makedirs(save_dir, exist_ok=True)

# Store the original savefig function from Figure class
original_fig_savefig = plt.Figure.savefig

@functools.wraps(original_fig_savefig)
def new_savefig(self, fname, *args, **kwargs):
    # Only modify if fname is a string (not a file object or figure)
    if isinstance(fname, str):
        # If it's not an absolute path, prepend our directory
        if not os.path.isabs(fname):
            fname = os.path.join(save_dir, fname)
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(fname), exist_ok=True)
    
    # Call original savefig
    return original_fig_savefig(self, fname, *args, **kwargs)

# Replace the Figure class's savefig method
plt.Figure.savefig = new_savefig

# Also patch pyplot.savefig
original_pyplot_savefig = plt.savefig

@functools.wraps(original_pyplot_savefig)
def new_pyplot_savefig(fname, *args, **kwargs):
    # Only modify if fname is a string
    if isinstance(fname, str):
        # If it's not an absolute path, prepend our directory
        if not os.path.isabs(fname):
            fname = os.path.join(save_dir, fname)
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(fname), exist_ok=True)
    
    # Call original savefig
    return original_pyplot_savefig(fname, *args, **kwargs)

plt.savefig = new_pyplot_savefig

print(f"✓ All figures will be saved to: {os.path.abspath(save_dir)}")

In [ ]:
# # dataset of all collected reviews
# reviews_df= pd.read_parquet("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/all_collected_reviews_25112025.parquet")
# reviews_df.info()

In [ ]:

# Check file size
file_path = "/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/all_collected_reviews_25112025.parquet"
print(f"File size: {os.path.getsize(file_path)} bytes")

# Try to read just metadata
try:
    metadata = pq.read_metadata(file_path)
    print("Metadata read successfully")
    print(f"Number of rows: {metadata.num_rows}")
    print(f"Number of columns: {metadata.num_columns}")
except Exception as e:
    print(f"Error reading metadata: {e}")

In [ ]:

try:
    reviews_df = pd.read_parquet(
        file_path,
        engine='fastparquet'  # Instead of pyarrow
    )
except Exception as e:
    print(f"Fastparquet error: {e}") 

In [ ]:
reviews_df.info()

In [ ]:
pools_df = pd.read_csv("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/swimmingpool_list_to_use_25112025.csv")
pools_df.drop(columns=['status', 'cluster_id', 'index_right', 'Unnamed: 54', 'Unnamed: 55', 'match_score', 'match_type'], inplace=True)
pools_df.info()

# Data distribution of  all reviews collected

In [ ]:
### aggregating the reviews by placeid
# Count collected reviews per place
reviews_per_place = (
    reviews_df
    .groupby("place_id", as_index=False)
    .agg(
        reviews_collected=("review_id", "count")
    )
)
reviews_per_place.info()

In [ ]:
#### merge with pools df
coverage_df = (
    pools_df
    .merge(reviews_per_place, on="place_id", how="right")
    .assign(
        reviews_collected=lambda d: d["reviews_collected"].fillna(0).astype(int),
        coverage_ratio=lambda d: np.where(
            d["total_reviews"] > 0,
            d["reviews_collected"] / d["total_reviews"],
            np.nan
        )
    )
)
coverage_df.info()

In [ ]:
coverage_df[coverage_df["WKR_NAME"] == 'Nürnberg-Süd']

In [ ]:
## pool per constitunency
pools_by_constituency = (
    coverage_df
    .groupby("WKR_NAME", as_index=False)
    .agg(
        n_pools=("place_id", "count")
    )
    .assign(
        pool_share=lambda d: d["n_pools"] / d["n_pools"].sum()
    )
    .sort_values("n_pools", ascending=False)
)
pools_by_constituency.info()

In [ ]:
#number of pools per state
pools_by_state = (
    coverage_df
    .groupby("LAND_NAME", as_index=False)
    .agg(
        n_pools=("place_id", "count")
    )
    .assign(
        pool_share=lambda d: d["n_pools"] / d["n_pools"].sum()
    )
    .sort_values("n_pools", ascending=False)
)

pools_by_state.info()

In [ ]:
# Review distribution across pools (global)

review_distribution_summary = (
    coverage_df
    .agg(
        n_pools=("place_id", "count"),
        total_reviews_collected=("reviews_collected", "sum"),
        mean_reviews=("reviews_collected", "mean"),
        median_reviews=("reviews_collected", "median"),
        max_reviews=("reviews_collected", "max"),
        mean_coverage=("coverage_ratio", "mean"),
        median_coverage=("coverage_ratio", "median")
    )
)
review_distribution_summary


In [ ]:
#review coverage by constituency
coverage_by_constituency = (
    coverage_df
    .groupby("WKR_NAME", as_index=False)
    .agg(
        n_pools=("place_id", "count"),
        total_reviews=("reviews_collected", "sum"),
        mean_reviews_per_pool=("reviews_collected", "mean"),
        median_reviews_per_pool=("reviews_collected", "median"),
        mean_coverage=("coverage_ratio", "mean"),
        median_coverage=("coverage_ratio", "median"),
        fully_covered_share=("coverage_ratio", lambda x: (x >= 0.8).mean())
    )
    .sort_values("n_pools", ascending=False)
)

coverage_by_constituency.info()

In [ ]:
#review coverage by state
coverage_by_state = (
    coverage_df
    .groupby("LAND_NAME", as_index=False)
    .agg(
        n_pools=("place_id", "count"),
        total_reviews=("reviews_collected", "sum"),
        mean_reviews_per_pool=("reviews_collected", "mean"),
        median_reviews_per_pool=("reviews_collected", "median"),
        mean_coverage=("coverage_ratio", "mean"),
        median_coverage=("coverage_ratio", "median"),
        fully_covered_share=("coverage_ratio", lambda x: (x >= 0.8).mean())
    )
    .sort_values("n_pools", ascending=False)
)
coverage_by_state.info()

In [ ]:
#Diagnostic columns for interpretation (VERY useful)
coverage_df = coverage_df.assign(
    review_volume_bin=pd.cut(
        coverage_df["total_reviews"],
        bins=[0, 10, 50, 100, 250, 500, np.inf],
        labels=["≤10", "11–50", "51–100", "101–250", "251–500", "500+"]
    ),
    coverage_class=pd.cut(
        coverage_df["coverage_ratio"],
        bins=[0, 0.25, 0.5, 0.75, 0.95, 1.01],
        labels=["<25%", "25–50%", "50–75%", "75–95%", "≈100%"]
    )
)
coverage_df.info()

# Visualizations of data distributions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")


In [ ]:
#Distribution of textual reviews per pool (skew check)
# Purpose:
# Shows whether a small number of pools dominate the review volume.

plt.figure(figsize=(8, 5))

sns.histplot(
    coverage_df["reviews_collected"],
    bins=50,
    log_scale=(True, False)
)

plt.xlabel("Textual reviews per pool (log scale)")
plt.ylabel("Number of pools")
plt.title("Distribution of Textual Reviews per Pool")

plt.tight_layout()
#plt.savefig("distribution_reviews_collected_per_pool.png")
plt.show()

In [ ]:
#Distribution of coverage ratios
# Purpose:
# Distinguishes partial collection from complete collection.

plt.figure(figsize=(8, 5))
sns.histplot(
    coverage_df["coverage_ratio"].dropna(),
    bins=30
)

plt.xlabel("Coverage ratio (collected / available reviews)")
plt.ylabel("Number of pools")
plt.title("Distribution of Review Coverage Across Pools")

plt.tight_layout()
plt.savefig("distribution_coverage_ratios.png")
plt.show()


In [ ]:
# #Collected vs available reviews (key diagnostic plot)
# Purpose:
# Visually separates:
# low-review but complete pools
# high-review but partially collected pools
# This is the most important figure.

plt.figure(figsize=(7, 7))

sns.scatterplot(
    data=coverage_df,
    x="total_reviews",
    y="reviews_collected",
    alpha=0.4,
    s=30
)

max_reviews = coverage_df["total_reviews"].max()

plt.plot(
    [0, max_reviews],
    [0, max_reviews],
    linestyle="--",
    color="red",
    label="Full coverage (y = x)"
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel("Total reviews available (log)")
plt.ylabel("Reviews collected (log)")
plt.title("Collected vs Available Reviews per Pool")
plt.legend()

plt.tight_layout()
plt.savefig("collected_vs_available_reviews.png")
plt.show()


In [ ]:
#Pools per state (coverage of places)
#Purpose:
#Shows spatial representativeness of pools.
plt.figure(figsize=(10, 5))

sns.barplot(
    data=coverage_by_state,
    x="LAND_NAME",
    y="n_pools"
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("State")
plt.ylabel("Number of pools")
plt.title("Number of Pools per State")

plt.tight_layout()
plt.savefig("pools_per_state.png")
plt.show()



In [ ]:
#Mean coverage ratio per state
#Purpose:
#Shows data quality differences across space.

plt.figure(figsize=(10, 5))

sns.barplot(
    data=coverage_by_state,
    x="LAND_NAME",
    y="mean_coverage"
)

plt.axhline(0.8, linestyle="--", color="red", label="80% coverage threshold")

plt.xticks(rotation=45, ha="right")
plt.xlabel("State")
plt.ylabel("Mean coverage ratio")
plt.title("Average Review Coverage per State")
plt.legend()

plt.tight_layout()
plt.savefig("mean_coverage_per_state.png")
plt.show()


In [ ]:
#6. Constituency-level coverage vs number of pools
# Purpose:
# Identifies constituencies that may bias results due to:
# few pools
# low coverage

plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=coverage_by_constituency,
    x="n_pools",
    y="mean_coverage",
    alpha=0.6
)

plt.xlabel("Number of pools in constituency")
plt.ylabel("Mean coverage ratio")
plt.title("Coverage vs Pool Count by Constituency")

plt.tight_layout()
plt.savefig("coverage_vs_poolcount_constituency.png")
plt.show()


In [ ]:
#Coverage class composition (stacked bar)
# Purpose:
# Shows how much of your dataset is fully vs partially collected.


coverage_class_dist = (
    coverage_df
    .groupby("coverage_class")
    .size()
    .reset_index(name="n_pools")
    .assign(share=lambda d: d["n_pools"] / d["n_pools"].sum())
)

plt.figure(figsize=(7, 5))

sns.barplot(
    data=coverage_class_dist,
    x="coverage_class",
    y="share"
)

plt.xlabel("Coverage class")
plt.ylabel("Share of pools")
plt.title("Coverage Completeness of Pools")

plt.tight_layout()
plt.savefig("coverage_completeness_pools.png")
plt.show()


In [ ]:
#(Optional but powerful) Review volume × coverage heatmap
# Purpose:
# Compact summary of where partial coverage concentrates.

heatmap_data = (
    coverage_df
    .groupby(["review_volume_bin", "coverage_class"])
    .size()
    .unstack(fill_value=0)
)

plt.figure(figsize=(9, 6))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Coverage class")
plt.ylabel("Review volume bin")
plt.title("Review Volume vs Coverage Completeness")

plt.tight_layout()
plt.savefig("review_volume_coverage_heatmap.png")
plt.show()


# Distribution of classified data 

About 700k of the 1.1 million reviews were classified successfully. Given that we may not bw able to reclassify the remaining 400 before the imminent deadline 15.01.2025, we need to check idf the distribution is good enough


In [ ]:
# Check file size
file_path = "/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/classified_reviews_05122025.parquet"
print(f"File size: {os.path.getsize(file_path)} bytes")

# Try to read just metadata
try:
    metadata = pq.read_metadata(file_path)
    print("Metadata read successfully")
    print(f"Number of rows: {metadata.num_rows}")
    print(f"Number of columns: {metadata.num_columns}")
except Exception as e:
    print(f"Error reading metadata: {e}")

In [ ]:
try:
    classified_df = pd.read_parquet(
        file_path,
        engine='fastparquet'  # Instead of pyarrow
    )
except Exception as e:
    print(f"Fastparquet error: {e}")

In [ ]:
# classified_df.drop(classified_df[~classified_df['review_id'].isin((reviews_df.loc[reviews_df['has_text'], 'review_id']).tolist())].index, inplace=True)
# classified_df.to_parquet("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/classified_reviews_05122025.parquet")

In [ ]:
#Define “has text” correctly (IMPORTANT)
#You should treat a review as textual if either original or 
# translated text exists and is non-empty.

reviews_df = reviews_df.assign(
    has_text=lambda d: (
        d["org_review"].notna() & d["org_review"].str.strip().ne("")
    ) | (
        d["trans_review"].notna() & d["trans_review"].str.strip().ne("")
    )
)


In [ ]:
#Count TEXT reviews per place (1.1M baseline)
#This gives:one row per place
#how many textual reviews exist, regardless of classification

text_reviews_per_place = (
    reviews_df
    .loc[reviews_df["has_text"]]
    .groupby("place_id", as_index=False)
    .agg(
        text_reviews_available=("review_id", "count")
    )
)

In [ ]:
#Link classified reviews back to places
classified_with_place = (
    classified_df[['review_id']]
    .merge(
        reviews_df[["review_id", "place_id"]],
        on="review_id",
        how="left",
        validate="many_to_one"
    )
)



In [ ]:
#If this fails, you have review IDs in classified_df that don’t exist in reviews_df.
assert classified_with_place["place_id"].notna().all()

In [ ]:
#Count CLASSIFIED text reviews per place (700k)
classified_reviews_per_place = (
    classified_with_place
    .groupby("place_id", as_index=False)
    .agg(
        text_reviews_classified=("review_id", "count")
    )
)


In [ ]:
#Build MASTER classification coverage table (place level)

#This is the table you will analyze and plot
#At this point, every pool has: how many text reviews exist,
# how many were classified,and the fraction classified

classification_coverage_df = (
    pools_df
    .merge(text_reviews_per_place, on="place_id", how="left")
    .merge(classified_reviews_per_place, on="place_id", how="left")
    .assign(
        text_reviews_available=lambda d: d["text_reviews_available"].fillna(0).astype(int),
        text_reviews_classified=lambda d: d["text_reviews_classified"].fillna(0).astype(int),
        classification_coverage=lambda d: np.where(
            d["text_reviews_available"] > 0,
            d["text_reviews_classified"] / d["text_reviews_available"],
            np.nan
        ),
        has_any_classified=lambda d: d["text_reviews_classified"] > 0
    )
)


## Key diagnostics (quick reality checks)


In [ ]:
# Totals should match what you expect
classification_coverage_df["text_reviews_available"].sum()
# should be ~1.1M

classification_coverage_df["text_reviews_classified"].sum()
# should be ~700k


In [ ]:
# How many pools are represented?

summary = {
    "total_pools": classification_coverage_df["place_id"].nunique(),
    "pools_with_text": (classification_coverage_df["text_reviews_available"] > 0).sum(),
    "pools_with_any_classified": classification_coverage_df["has_any_classified"].sum(),
}
summary["share_text_pools_covered"] = (
    summary["pools_with_any_classified"] / summary["pools_with_text"]
)

summary


## Visualisation

In [ ]:
# PER POOL: Classification coverage distribution
# 1.1 Histogram of classification coverage per pool
# What this shows:
# Most pools are partially but substantially classified; very few are zeroed out.
# Interpretation (paper-ready):

# A strong mass away from zero indicates classification failures are review-level, not place-level.

# Near-zero pools are rare (≈2%)

plt.figure(figsize=(18, 16))

sns.histplot(
    classification_coverage_df
    .loc[classification_coverage_df["text_reviews_available"] > 0, "classification_coverage"],
    bins=30
)

plt.xlabel("Fraction of text reviews classified", fontsize=30)
plt.xticks(fontsize=30)
plt.ylabel("Number of pools", fontsize=30)
plt.yticks(fontsize=30)
#plt.title("Classification Coverage per Pool", fontsize=30)

plt.tight_layout()
plt.savefig("classification_coverage_per_pool.png")
plt.show()


In [ ]:
# Text reviews vs classified reviews per pool (log–log)

# This is the single most important diagnostic.
# Why this matters:
# Points hug the diagonal → failures are proportional, not selective
# No “horizontal bands” → pools are not being dropped wholesale
# Confirms random-like truncation, not structural bias

plt.figure(figsize=(7, 7))

sns.scatterplot(
    data=classification_coverage_df,
    x="text_reviews_available",
    y="text_reviews_classified",
    alpha=0.4,
    s=25
)

max_val = classification_coverage_df["text_reviews_available"].max()

plt.plot(
    [1, max_val],
    [1, max_val],
    linestyle="--",
    color="red",
    label="Full classification (y = x)"
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel("Text reviews available (log)")
plt.ylabel("Text reviews classified (log)")
plt.title("Textual Reviews vs Classified Reviews per Pool")
plt.legend()

plt.tight_layout()
plt.savefig("textual_vs_classified_reviews_per_pool.png")
plt.show()


In [ ]:
# PER CONSTITUENCY: Structural audit
# 2Constituency-level aggregation
constituency_coverage = (
    classification_coverage_df
    .groupby("WKR_NAME", as_index=False)
    .agg(
        text_reviews=("text_reviews_available", "sum"),
        classified_reviews=("text_reviews_classified", "sum"),
        n_pools=("place_id", "count"),
        pools_with_classified=("has_any_classified", "sum")
    )
    .assign(
        classification_share=lambda d: d["classified_reviews"] / d["classified_reviews"].sum(),
        text_share=lambda d: d["text_reviews"] / d["text_reviews"].sum(),
        pool_coverage_share=lambda d: d["pools_with_classified"] / d["n_pools"]
    )
)


In [ ]:
# Text share vs classified share (constituencies)
# This directly tests spatial bias
# Interpretation:
# Tight clustering along the diagonal = no constituency-level bias
# Outliers (if any) can be explicitly discussed

plt.figure(figsize=(7, 7))

plt.scatter(
    constituency_coverage["text_share"],
    constituency_coverage["classification_share"],
    alpha=0.6
)

max_val = max(
    constituency_coverage["text_share"].max(),
    constituency_coverage["classification_share"].max()
)

plt.plot([0, max_val], [0, max_val], linestyle="--", color="red")

plt.xlabel("Share of textual reviews")
plt.ylabel("Share of classified reviews")
plt.title("Constituency-Level Representativeness of Classified Reviews")

plt.tight_layout()
plt.savefig("constituency_level_representativeness.png")
plt.show()


In [ ]:
#Pool representation by constituency
# What this tells reviewers:
# Most constituencies retain near-complete pool representation
# Failures do not cluster geographically

plt.figure(figsize=(10, 5))

sns.histplot(
    constituency_coverage["pool_coverage_share"],
    bins=20
)

plt.axvline(0.8, linestyle="--", color="red", label="80% threshold")

plt.xlabel("Share of pools with ≥1 classified review")
plt.ylabel("Number of constituencies")
plt.title("Pool Representation Across Constituencies")
plt.legend()

plt.tight_layout()
plt.savefig("pool_representation_constituency.png")
plt.show()


In [ ]:
#PER STATE: High-level spatial robustness
# 3.1 State-level aggregation

state_coverage = (
    classification_coverage_df
    .groupby("LAND_NAME", as_index=False)
    .agg(
        text_reviews=("text_reviews_available", "sum"),
        classified_reviews=("text_reviews_classified", "sum"),
        n_pools=("place_id", "count"),
        pools_with_classified=("has_any_classified", "sum")
    )
    .assign(
        classification_share=lambda d: d["classified_reviews"] / d["classified_reviews"].sum(),
        text_share=lambda d: d["text_reviews"] / d["text_reviews"].sum(),
        pool_coverage_share=lambda d: d["pools_with_classified"] / d["n_pools"]
    )
)


In [ ]:
#Text vs classified share (states)
# This is the “no-spatial-bias” figure.
plt.figure(figsize=(7, 7))

plt.scatter(
    state_coverage["text_share"],
    state_coverage["classification_share"],
    s=80
)

max_val = max(
    state_coverage["text_share"].max(),
    state_coverage["classification_share"].max()
)

plt.plot([0, max_val], [0, max_val], linestyle="--", color="red")

plt.xlabel("Share of textual reviews")
plt.ylabel("Share of classified reviews")
plt.title("State-Level Representativeness of Classified Reviews")

plt.tight_layout()
plt.savefig("state_level_representativeness.png")
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=state_coverage,
    x="LAND_NAME",
    y="pool_coverage_share"
)

plt.axhline(0.9, linestyle="--", color="red", label="90% threshold")

plt.xticks(rotation=45, ha="right")
plt.ylabel("Share of pools with ≥1 classified review")
plt.title("Pool Coverage After Classification by State")
plt.legend()

plt.tight_layout()
plt.savefig("pool_coverage_state.png")
plt.show()


# statistical testing

In [ ]:
from scipy.stats import chi2_contingency

A1. χ² test: Spatial representativeness (State / Constituency)
Null hypothesis

The distribution of classified reviews across space is the same as that of all textual reviews.
Interpretation

p > 0.05 → fail to reject null → no spatial bias

p ≤ 0.05 → potential spatial distortion (you’d investigate outliers)

With your coverage numbers, it is very likely p ≫ 0.05.

In [ ]:
state_contingency = (
    state_coverage
    .loc[:, ["text_reviews", "classified_reviews"]]
    .set_index(state_coverage["LAND_NAME"])
)

chi2_state, p_state, dof_state, _ = chi2_contingency(state_contingency)

chi2_state, p_state


In [ ]:
#Constituency-level χ² test (stronger, harsher)
# Why this is impressive to reviewers:
# 299 constituencies = very high-resolution spatial test.
constituency_contingency = (
    constituency_coverage
    .loc[:, ["text_reviews", "classified_reviews"]]
    .set_index(constituency_coverage["WKR_NAME"])
)

chi2_const, p_const, _, _ = chi2_contingency(constituency_contingency)

chi2_const, p_const


A2. KS test: Pool-level structural preservation

Now we test whether the distribution of review volume per pool changes after classification.

Interpretation

p > 0.05 → distributions are statistically indistinguishable

Even p < 0.05 with very small KS statistic is acceptable at this scale

Large N inflates sensitivity

Report effect size (KS statistic), not just p

Example acceptable language:

“Although the KS test is statistically significant due to large sample size, the KS statistic (D = 0.06) indicates only minor distributional differences.”

In [ ]:
from scipy.stats import ks_2samp


In [ ]:
per_pool_text = (
    classification_coverage_df
    .loc[classification_coverage_df["text_reviews_available"] > 0,
         "text_reviews_available"]
)

per_pool_classified = (
    classification_coverage_df
    .loc[classification_coverage_df["text_reviews_classified"] > 0,
         "text_reviews_classified"]
)


In [ ]:
#KS test
ks_stat, ks_p = ks_2samp(
    per_pool_text,
    per_pool_classified,
    alternative="two-sided",
    mode="asymp"
)

ks_stat, ks_p


A3. Are high-review pools selectively under-classified?

This tests non-random truncation.

Compare coverage ratios across review-volume quartiles

In [ ]:
#Compare coverage ratios across review-volume quartiles
classification_coverage_df["volume_quartile"] = pd.qcut(
    classification_coverage_df["text_reviews_available"],
    q=4,
    duplicates="drop"
)


In [ ]:
coverage_by_volume = (
    classification_coverage_df
    .groupby("volume_quartile")
    .agg(
        mean_coverage=("classification_coverage", "mean"),
        median_coverage=("classification_coverage", "median"),
        n_pools=("place_id", "count")
    )
)

coverage_by_volume


In [ ]:
#Optional non-parametric test (robust):
from scipy.stats import kruskal

groups = [
    g["classification_coverage"].dropna().values
    for _, g in classification_coverage_df.groupby("volume_quartile")
]

kruskal_stat, kruskal_p = kruskal(*groups)

kruskal_stat, kruskal_p


## 2. Extending the audit to your precise concern
(“fractional sidelining within constituencies & states”)

Now we formalize your intuition.

2.1 Fractional coverage distribution within constituencies

You want to know:

Do some constituencies contain pools that are systematically under-classified

In [ ]:
constituency_fractional = (
    classification_coverage_df
    .loc[classification_coverage_df["text_reviews_available"] > 0]
    .groupby("WKR_NAME")
    .agg(
        mean_frac=("classification_coverage", "mean"),
        median_frac=("classification_coverage", "median"),
        p10=("classification_coverage", lambda x: x.quantile(0.10)),
        p90=("classification_coverage", lambda x: x.quantile(0.90)),
        n_pools=("place_id", "count")
    )
)


In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(constituency_fractional["mean_frac"], bins=30)

plt.axvline(
    classification_coverage_df["classification_coverage"].mean(),
    linestyle="--", color="red", label="Global mean"
)

plt.xlabel("Mean classification coverage (per constituency)")
plt.ylabel("Number of constituencies")
plt.title("Distribution of Fractional Classification Coverage Across Constituencies")
plt.legend()
plt.tight_layout()
plt.savefig("distribution_fractional_classification_coverage_constituencies.png")
plt.show()


2.2 Worst-case analysis (reviewer-proof)

Identify potentially sidelined pools:

In [ ]:
classification_coverage_df["low_coverage_flag"] = (
    classification_coverage_df["classification_coverage"] < 0.3
)


In [ ]:
 # Now check spatial concentration:
low_coverage_by_state = (
    classification_coverage_df
    .groupby("LAND_NAME")
    .agg(
        low_cov_share=("low_coverage_flag", "mean"),
        n_pools=("place_id", "count")
    )
)
#If no state spikes → no spatial sidelining
plt.figure(figsize=(10, 5))
sns.barplot(data=low_coverage_by_state, x="LAND_NAME", y="low_cov_share")
plt.axhline(0.1, linestyle="--", color="red", label="10% threshold")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Share of low-coverage pools")
plt.title("Low-Coverage Pools by State")
plt.legend()
plt.tight_layout()
plt.savefig("low_coverage_pools_by_state.png")
plt.show()

# Validating the classification of the 700k

In [ ]:
reviews_collected_df = reviews_df[reviews_df['org_review'].notnull() & (reviews_df['trans_review'].notnull() )]

In [ ]:
text_reviews_path = "/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/text_reviews_summarized_25112025.parquet"

try:
    text_reviews = pd.read_parquet(
        text_reviews_path,
        engine='fastparquet'  # Instead of pyarrow
    )
except Exception as e:
    print(f"Fastparquet error: {e}")

In [ ]:
text_reviews.info()

In [ ]:
classified_val_df = (
    classified_df
    .merge(reviews_df[["review_id", "place_id", "trans_review"]], on="review_id", how="left")
    .merge(text_reviews[["review_id", "summary"]], on="review_id", how="left")
)

classified_val_df.info()


In [ ]:
classified_val_df.nunique()

In [ ]:
#checking the labels
#for the  various labels of social encounter what is thhr distribution of the  other labels
classified_val_df.groupby('social_encounter').agg({'racial_connotations':'value_counts',
'connotation_type':'value_counts'})

In [ ]:
#clean up because it appears the LLm labelled individually not as a hierarchy
#if the social encounter is 'no' the racial connotations  and connatation_type should be N/A whicxh is not applicable
classified_val_df.loc[classified_val_df['social_encounter']=='no', ['racial_connotations', 'connotation_type']] = 'N/A'
#if the racial connotations is 'no' then the connotation_type should be N/A
classified_val_df.loc[classified_val_df['racial_connotations']=='no', ['connotation_type']] = 'N/A'
# for the  other labels in the connotation type other than  plaseant , ambiguous, discriminatory change these three
classified_val_df.loc[classified_val_df['connotation_type']== 'unpleasant', ['connotation_type']] = 'discriminatory'
classified_val_df.loc[classified_val_df['connotation_type']== 'neutral', ['connotation_type']] = 'ambiguous'
#

In [ ]:
2219 + 2734

In [ ]:
#check again
classified_val_df.groupby('social_encounter').agg({'racial_connotations':'value_counts',
'connotation_type':'value_counts'})

In [ ]:
classified_val_df.connotation_type.unique()

In [ ]:
classified_val_df.racial_connotations.unique()

In [ ]:
classified_val_df.social_encounter.unique()

In [ ]:
##save the cleaned classified val df
classified_val_df.to_parquet("/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/classified_reviews_cleaned_06012026.parquet")

# Select the reviews to validate

In [ ]:
import pandas as pd
import numpy as np
import random
from collections import defaultdict

def create_stratified_sample(df, sample_sizes, random_seed=42):
    """
    Create a stratified random sample based on hierarchical labels.
    
    Parameters:
    -----------
    df : pandas DataFrame
        DataFrame with columns: 'review_id', 'trans_review', 'social_encounter', 
        'racial_connotations', 'connotation_type'
    sample_sizes : dict
        Dictionary with stratum names as keys and sample sizes as values
    random_seed : int
        Random seed for reproducibility
        
    Returns:
    --------
    pandas DataFrame: Stratified sample
    dict: Actual sample counts for each stratum
    """
    
    # Set random seed for reproducibility
    np.random.seed(random_seed)
    random.seed(random_seed)
    
    # Create stratum identifier
    def create_stratum(row):
        if row['social_encounter'] == 'no':
            return 'SE_no'
        elif row['social_encounter'] == 'yes' and row['racial_connotations'] == 'no':
            return 'SE_yes_RC_no'
        elif row['social_encounter'] == 'yes' and row['racial_connotations'] == 'yes':
            if row['connotation_type'] == 'pleasant':
                return 'SE_yes_RC_yes_pleasant'
            elif row['connotation_type'] == 'ambiguous':
                return 'SE_yes_RC_yes_ambiguous'
            elif row['connotation_type'] == 'discriminatory':
                return 'SE_yes_RC_yes_discriminatory'
        return 'other'
    
    # Add stratum column
    df = df.copy()
    df['stratum'] = df.apply(create_stratum, axis=1)
    
    # Initialize results
    sampled_data = []
    actual_counts = {}
    
    # Sample from each stratum
    for stratum, target_n in sample_sizes.items():
        stratum_data = df[df['stratum'] == stratum].copy()
        stratum_count = len(stratum_data)
        
        actual_counts[stratum] = {
            'available': stratum_count,
            'target': target_n,
            'sampled': 0
        }
        
        if stratum_count == 0:
            print(f"Warning: No data found for stratum {stratum}")
            continue
            
        # Sample from stratum
        if stratum_count <= target_n:
            # Take all if stratum has fewer items than target
            sample = stratum_data
            actual_counts[stratum]['sampled'] = stratum_count
            print(f"Stratum '{stratum}': Sampled all {stratum_count} (less than target {target_n})")
        else:
            # Random sample without replacement
            sample = stratum_data.sample(n=target_n, random_state=random_seed)
            actual_counts[stratum]['sampled'] = target_n
            print(f"Stratum '{stratum}': Sampled {target_n} out of {stratum_count}")
        
        # Add stratum identifier and sample to results
        sample['sampling_stratum'] = stratum
        sampled_data.append(sample)
    
    # Combine all samples
    if sampled_data:
        stratified_sample = pd.concat(sampled_data, ignore_index=True)
        
        # Shuffle the final sample to mix strata
        stratified_sample = stratified_sample.sample(frac=1, random_state=random_seed).reset_index(drop=True)
        
        # Add a sample ID for tracking
        stratified_sample['sample_id'] = range(1, len(stratified_sample) + 1)
        
        return stratified_sample, actual_counts
    else:
        return pd.DataFrame(), actual_counts


def analyze_strata_distribution(df):
    """
    Analyze the distribution of reviews across hierarchical strata.
    
    Parameters:
    -----------
    df : pandas DataFrame
        DataFrame with label columns
        
    Returns:
    --------
    pandas DataFrame: Counts and percentages for each stratum
    """
    
    # Create stratum for all data
    def create_stratum(row):
        if row['social_encounter'] == 'no':
            return 'SE_no'
        elif row['social_encounter'] == 'yes' and row['racial_connotations'] == 'no':
            return 'SE_yes_RC_no'
        elif row['social_encounter'] == 'yes' and row['racial_connotations'] == 'yes':
            if row['connotation_type'] == 'pleasant':
                return 'SE_yes_RC_yes_pleasant'
            elif row['connotation_type'] == 'ambiguous':
                return 'SE_yes_RC_yes_ambiguous'
            elif row['connotation_type'] == 'discriminatory':
                return 'SE_yes_RC_yes_discriminatory'
        return 'other'
    
    df = df.copy()
    df['stratum'] = df.apply(create_stratum, axis=1)
    
    # Calculate counts and percentages
    stratum_counts = df['stratum'].value_counts().reset_index()
    stratum_counts.columns = ['stratum', 'count']
    stratum_counts['percentage'] = (stratum_counts['count'] / len(df) * 100).round(2)
    
    # Calculate cumulative percentage
    stratum_counts = stratum_counts.sort_values('count', ascending=False)
    stratum_counts['cumulative_percentage'] = stratum_counts['percentage'].cumsum()
    
    return stratum_counts


def generate_sample_plan(total_n=400, strategy='rare_focused'):
    """
    Generate a sample plan based on different strategies.
    
    Parameters:
    -----------
    total_n : int
        Total sample size
    strategy : str
        'balanced': Equal-ish sample from each relevant stratum
        'rare_focused': Focus on rare categories
        'proportional': Proportional to population (not recommended for validation)
        
    Returns:
    --------
    dict: Sample sizes for each stratum
    """
    
    if strategy == 'balanced':
        # Balanced approach as discussed
        return {
            'SE_no': 100,  # social_encounter=no
            'SE_yes_RC_no': 100,  # social_encounter=yes, racial_connotations=no
            'SE_yes_RC_yes_pleasant': 70,
            'SE_yes_RC_yes_ambiguous': 70,
            'SE_yes_RC_yes_discriminatory': 60
        }
    
    elif strategy == 'rare_focused':
        # Focus more on rare categories
        return {
            'SE_no': 80,
            'SE_yes_RC_no': 80,
            'SE_yes_RC_yes_pleasant': 60,
            'SE_yes_RC_yes_ambiguous': 90,  # More for ambiguous (often challenging)
            'SE_yes_RC_yes_discriminatory': 90  # More for rare but critical
        }
    
    elif strategy == 'proportional':
        # This would need actual data to calculate proportions
        # Placeholder - you'd need to run analyze_strata_distribution first
        print("Warning: Proportional sampling not recommended for validation")
        print("You need to calculate proportions from your data first")
        return None


def export_sample_for_validation(sample_df, output_path):
    """
    Export sample in a format convenient for manual validation.
    
    Parameters:
    -----------
    sample_df : pandas DataFrame
        Stratified sample
    output_path : str
        Path to save the validation file
    """
    
    # Select and reorder columns for validation
    validation_cols = [
        'sample_id', 'sampling_stratum','review_id', 'trans_review', 'summary',
        'social_encounter', 'racial_connotations', 'connotation_type',
        'human_social_encounter', 'human_racial_connotation', 
        'human_connotation_type', 'notes'
    ]
    
    # Create empty columns for human validation
    for col in ['human_social_encounter', 'human_racial_connotation', 
                'human_connotation_type', 'notes']:
        if col not in sample_df.columns:
            sample_df[col] = ''
    
    # Export to Excel for easy manual validation
    sample_df[validation_cols].to_excel(output_path, index=False)
    print(f"Validation file saved to: {output_path}")
    
    # Also create a summary sheet
    with pd.ExcelWriter(output_path, engine='openpyxl', mode='a') as writer:
        # Create summary statistics
        summary = pd.DataFrame({
            'Statistic': ['Total Sample Size', 'Number of Strata', 
                         'Date Created', 'Random Seed Used'],
            'Value': [len(sample_df), sample_df['sampling_stratum'].nunique(),
                     pd.Timestamp.now().strftime('%Y-%m-%d'), 42]
        })
        summary.to_excel(writer, sheet_name='Sample_Info', index=False)


# ==================== USAGE EXAMPLE ====================

def main():
    # Load your data (replace with your actual data loading)
    # Assuming you have a CSV with the LLM classifications
    df = classified_val_df.copy()
    print("Data loaded. Number of reviews:", len(df))
    
    # Analyze the distribution in your full dataset
    print("\n=== Analyzing full dataset distribution ===")
    distribution = analyze_strata_distribution(df)
    print(distribution)
    
    # Generate sample plan
    print("\n=== Generating sample plan ===")
    sample_plan = generate_sample_plan(total_n=400, strategy='rare_focused')
    print("Sample plan:")
    for stratum, n in sample_plan.items():
        print(f"  {stratum}: {n}")
    
    # Create stratified sample
    print("\n=== Creating stratified sample ===")
    stratified_sample, actual_counts = create_stratified_sample(df, sample_plan, random_seed=42)
    
    print(f"\nTotal sampled reviews: {len(stratified_sample)}")
    print("\nActual sampling results:")
    for stratum, counts in actual_counts.items():
        print(f"{stratum}: {counts['sampled']} sampled (target: {counts['target']}, available: {counts['available']})")
    
    # Verify the sample distribution
    print("\n=== Sample distribution ===")
    sample_dist = stratified_sample['sampling_stratum'].value_counts()
    print(sample_dist)
    
    # Export for manual validation
    print("\n=== Exporting for validation ===")
    export_sample_for_validation(stratified_sample, '/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/validating_classification/validation_sample.xlsx')
    
    # Save the sample for later use
    stratified_sample.to_csv('/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/validating_classification/stratified_sample.csv', index=False)
    print("\nSample saved to 'stratified_sample.csv'")
    
    # Optional: Save sampling metadata
    metadata = {
        'total_population': len(df),
        'sample_size': len(stratified_sample),
        'strata_counts': actual_counts,
        'sampling_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'random_seed': 42,
        'sample_plan': sample_plan
    }
    
    import json
    with open('/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/validating_classification/sampling_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print("Sampling metadata saved to 'sampling_metadata.json'")


if __name__ == "__main__":
    main()

# selecting the racial reviews from the 700k classified

In [ ]:
classified_val_df[classified_val_df['racial_connotations']=='yes'].reset_index(drop=True).to_csv('/Users/i2scg/Documents/workstation_repos_30122025/ICWSM_15012026/working_data/racial_theme_analysis_20260106_221456/racial_reviews_700k.csv')

# Checking the validated stratified sample

In [ ]:
def calculate_accuracy_metrics(validation_df):
    """
    Calculate accuracy metrics from validation results.
    
    Parameters:
    -----------
    validation_df : pandas DataFrame
        DataFrame with both LLM and human labels
        
    Returns:
    --------
    dict: Accuracy metrics at each hierarchical level
    """
    
    # Filter out rows without human validation
    validated = validation_df[
        (validation_df['human_social_encounter'] != '') & 
        (validation_df['human_social_encounter'].notna())
    ].copy()
    
    if len(validated) == 0:
        print("No validated data found")
        return {}
    
    metrics = {}
    
    # 1. Overall Social Encounter Accuracy
    se_correct = (validated['social_encounter'] == validated['human_social_encounter']).sum()
    metrics['social_encounter_accuracy'] = {
        'correct': int(se_correct),
        'total': len(validated),
        'accuracy': se_correct / len(validated)
    }
    
    # 2. Conditional Racial Connotation Accuracy (only where social_encounter=yes)
    se_yes_mask = validated['human_social_encounter'] == 'yes'
    se_yes_data = validated[se_yes_mask]
    
    if len(se_yes_data) > 0:
        rc_correct = (se_yes_data['racial_connotation'] == se_yes_data['human_racial_connotation']).sum()
        metrics['racial_connotation_accuracy'] = {
            'correct': int(rc_correct),
            'total': len(se_yes_data),
            'accuracy': rc_correct / len(se_yes_data)
        }
        
        # 3. Conditional Connotation Type Accuracy (only where racial_connotation=yes)
        rc_yes_mask = se_yes_data['human_racial_connotation'] == 'yes'
        rc_yes_data = se_yes_data[rc_yes_mask]
        
        if len(rc_yes_data) > 0:
            ct_correct = (rc_yes_data['connotation_type'] == rc_yes_data['human_connotation_type']).sum()
            metrics['connotation_type_accuracy'] = {
                'correct': int(ct_correct),
                'total': len(rc_yes_data),
                'accuracy': ct_correct / len(rc_yes_data)
            }
    
    # 4. Full-path accuracy
    full_correct = (
        (validated['social_encounter'] == validated['human_social_encounter']) &
        (validated['racial_connotation'] == validated['human_racial_connotation']) &
        (validated['connotation_type'] == validated['human_connotation_type'])
    ).sum()
    
    metrics['full_path_accuracy'] = {
        'correct': int(full_correct),
        'total': len(validated),
        'accuracy': full_correct / len(validated)
    }
    
    # 5. Accuracy by stratum
    metrics['by_stratum'] = {}
    for stratum in validated['sampling_stratum'].unique():
        stratum_data = validated[validated['sampling_stratum'] == stratum]
        stratum_correct = (
            (stratum_data['social_encounter'] == stratum_data['human_social_encounter']) &
            (stratum_data['racial_connotation'] == stratum_data['human_racial_connotation']) &
            (stratum_data['connotation_type'] == stratum_data['human_connotation_type'])
        ).sum()
        
        metrics['by_stratum'][stratum] = {
            'correct': int(stratum_correct),
            'total': len(stratum_data),
            'accuracy': stratum_correct / len(stratum_data) if len(stratum_data) > 0 else 0
        }
    
    return metrics


def calculate_confidence_intervals(metrics):
    """
    Calculate Wilson score confidence intervals for accuracy metrics.
    
    Parameters:
    -----------
    metrics : dict
        Accuracy metrics from calculate_accuracy_metrics
        
    Returns:
    --------
    dict: Metrics with confidence intervals
    """
    
    from statsmodels.stats.proportion import proportion_confint
    
    metrics_with_ci = metrics.copy()
    
    for key in ['social_encounter_accuracy', 'racial_connotation_accuracy', 
                'connotation_type_accuracy', 'full_path_accuracy']:
        if key in metrics:
            correct = metrics[key]['correct']
            total = metrics[key]['total']
            accuracy = metrics[key]['accuracy']
            
            if total > 0:
                ci_low, ci_high = proportion_confint(correct, total, method='wilson')
                metrics_with_ci[key]['ci_95_low'] = ci_low
                metrics_with_ci[key]['ci_95_high'] = ci_high
                metrics_with_ci[key]['accuracy_formatted'] = f"{accuracy:.1%} ({ci_low:.1%} - {ci_high:.1%})"
    
    return metrics_with_ci


def print_validation_report(metrics_with_ci):
    """
    Print a formatted validation report.
    """
    print("=" * 60)
    print("VALIDATION REPORT")
    print("=" * 60)
    
    for key in ['social_encounter_accuracy', 'racial_connotation_accuracy', 
                'connotation_type_accuracy', 'full_path_accuracy']:
        if key in metrics_with_ci:
            m = metrics_with_ci[key]
            print(f"\n{key.replace('_', ' ').title()}:")
            print(f"  Correct: {m['correct']} / {m['total']}")
            print(f"  Accuracy: {m.get('accuracy_formatted', f'{m[\"accuracy\"]:.1%}')}")
    
    print("\n" + "=" * 60)
    print("ACCURACY BY STRATUM:")
    print("=" * 60)
    
    if 'by_stratum' in metrics_with_ci:
        for stratum, m in metrics_with_ci['by_stratum'].items():
            print(f"\n{stratum}:")
            print(f"  Correct: {m['correct']} / {m['total']}")
            print(f"  Accuracy: {m['accuracy']:.1%}")